# BindCraft: Protein binder design

> **Version 1.9.0.** The current version is always the one at
> [the releases page](https://github.com/a-LittleMonk/BindCraft-GoogleColab/releases).
> If that page shows a higher number than the line you are reading, this copy
> is out of date — [open the current notebook](https://colab.research.google.com/github/a-LittleMonk/BindCraft-GoogleColab/blob/main/BindCraft_colab.ipynb) and *File > Save a
> copy in Drive* to replace it. A saved copy never updates itself.

<img src="https://github.com/martinpacesa/BindCraft/blob/main/pipeline.png?raw=true">

Simple binder design pipeline using AlphaFold2 backpropagation, MPNN, and PyRosetta. Select your target and let the script do the rest of the work and finish once you have enough designs to order!

The designs will be saved on your Google Drive under BindCraft/[design_name]/ and you can continue running the design pipeline if the session times out and it will continue adding new designs.
---

**This copy is patched for current Colab (Python 3.13 / new JAX).** Changes from upstream:

* **Cell 8 &mdash; Compatibility patches (required).** Shims for `jnp.clip` and `clear_mem`, whose APIs were removed. Without these the pipeline cannot run.
* **Cell 9 &mdash; Tuning controls.** A `preset` dropdown at the top (**Production defaults** / **Fast debug** / **Custom**) plus 14 fields for values that upstream hardcodes, including the trajectory pLDDT gate and the CA-clash limit. Select *Production defaults* to reset everything.
* **Cell 11** fixes a bug where `binder_name` was overwritten by the loop comparing against it, which broke every rank past the first.
* **Cells 13 and 14** no longer crash when nothing has been ranked yet.

Run cells **1 to 14 in order**. Cells 8 and 9 must come after cell 6, and they will tell you if they don't.


In [ ]:
#@title 1. Installation
%%time
import os, time, gc, io
import contextlib
import json
from datetime import datetime
from ipywidgets import HTML, VBox
from IPython.display import display

if not os.path.isfile("bindcraft/params/done.txt"):
  print("Installing required BindCraft components")

  print("Pulling BindCraft code from Github")
  os.makedirs('/content/bindcraft/', exist_ok=True)
  !git clone https://github.com/martinpacesa/BindCraft /content/bindcraft/
  os.system("chmod +x /content/bindcraft/functions/dssp")
  os.system("chmod +x /content/bindcraft/functions/DAlphaBall.gcc")

  print("Installing ColabDesign")
  os.system("(mkdir bindcraft/params; apt-get install aria2 -qq; \
  aria2c -q -x 16 https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar; \
  tar -xf alphafold_params_2022-12-06.tar -C bindcraft/params; touch bindcraft/params/done.txt )&")
  os.system("pip install git+https://github.com/sokrypton/ColabDesign.git")
  # for debugging purposes
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabdesign colabdesign")

  # download params
  if not os.path.isfile("bindcraft/params/done.txt"):
    print("downloading AlphaFold params")
    while not os.path.isfile("bindcraft/params/done.txt"):
      time.sleep(5)

  print("BindCraft components installed")
else:
  print("BindCraft components already installed")

# PyRosetta is handled outside the block above, on purpose.
#
# Upstream installed it from west.rosettacommons.org only. That host is the IPD
# file server at UW, and when it is down pip retries five times, gives up, and
# the cell still announces "ready to run" - the failure only surfaces at cell 7.
# Worse, by then the AlphaFold params marker exists, so re-running this cell
# skipped the whole block and never retried, making a passing outage look
# permanent.
#
# pyrosetta-installer knows both official mirrors, so try the west one and fall
# back to Johns Hopkins. Checking find_spec rather than importing keeps this
# quick; importing PyRosetta itself takes a while.
def _have_pyrosetta():
  return os.system('python -c "import importlib.util as u, sys; sys.exit(0 if u.find_spec(\'pyrosetta\') else 1)" > /dev/null 2>&1') == 0

if _have_pyrosetta():
  print("PyRosetta already installed")
else:
  print("Installing PyRosetta")
  os.system("pip install -q pyrosetta-installer")
  for _mirror, _host in ((0, "west.rosettacommons.org"), (1, "graylab.jhu.edu")):
    print(f"  trying {_host}")
    os.system('python -c "import pyrosetta_installer; pyrosetta_installer.install_pyrosetta(mirror=%s, serialization=True)"' % _mirror)
    if _have_pyrosetta():
      break

if not _have_pyrosetta():
  raise RuntimeError(
      "PyRosetta did not install: both official mirrors failed to answer. "
      "Cell 7 needs it, so nothing below this will work. That is almost always "
      "a server outage rather than anything on your side - wait, then re-run "
      "this cell. It retries the install on its own now, so there is no need to "
      "reset the runtime.")

print("BindCraft installation is finished, ready to run!")

In [ ]:
#@title 2. Mount your Google Drive to save design results
from google.colab import drive
drive.mount('/content/drive')
currenttime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Google drive mounted at: {currenttime}")

bindcraft_google_drive = '/content/drive/My Drive/BindCraft/'
os.makedirs(bindcraft_google_drive, exist_ok=True)
print("BindCraft folder successfully created in your drive!")

In [ ]:
#@title 3. Binder design settings
# @markdown ---
# @markdown Enter path where to save your designs. We recommend to save on Google drive so that you can continue generating at any time.
design_path = "/content/drive/MyDrive/BindCraft/your_folder_Name/" # @param {"type":"string","placeholder":"/content/drive/MyDrive/BindCraft/PDL1/"}

# @markdown Enter the name that should be prefixed to your binders (generally target name).
binder_name = "TGFb1" # @param {"type":"string","placeholder":"PDL1"}

# @markdown The path to the .pdb structure of your target. Can be an experimental or AlphaFold2 structure. We recommend trimming the structure to as small as needed, as the whole selected chains will be backpropagated through the network and can significantly increase running times.
starting_pdb = "/content/drive/MyDrive/pdb_file_name.pdb" # @param {"type":"string","placeholder":"/content/bindcraft/example/PDL1.pdb"}

# @markdown Which chains of your PDB to target? Can be one or multiple, in a comma-separated format. Other chains will be ignored during design.
chains = "A" # @param {"type":"string","placeholder":"A,C"}

# @markdown What positions to target in your protein of interest? For example `1,2-10` or chain specific `A1-10,B1-20` or entire chains `A`. If left blank, an appropriate site will be selected by the pipeline.
target_hotspot_residues = "" # @param {"type":"string","placeholder":""}

# @markdown What is the minimum and maximum size of binders you want to design? Pipeline will randomly sample different sizes between these values.
lengths = "25,40" # @param {"type":"string","placeholder":"70,150"}

# @markdown How many binder designs passing filters do you require?
number_of_final_designs = 10 # @param {"type":"integer","placeholder":"100"}
# @markdown ---
# @markdown Enter path on your Google drive (/content/drive/MyDrive/BindCraft/[binder_name].json) to previous target settings to continue design campaign. If left empty, it will use the settings above and generate a new settings json in your design output folder.
load_previous_target_settings = "" # @param {"type":"string","placeholder":""}
# @markdown ---

if load_previous_target_settings:
    target_settings_path = load_previous_target_settings
else:
    # --- check the settings before anything is written -------------------
    # BindCraft does not open starting_pdb until the end of the first
    # trajectory, so a wrong path costs about half an hour before it says so.
    # Everything here is checked up front instead, and nothing is created
    # until it all passes.
    problems = []

    if "your_folder_Name" in design_path:
        problems.append("design_path is still the placeholder - point it at a folder on your Drive")
    if "pdb_file_name" in starting_pdb:
        problems.append("starting_pdb is still the placeholder - point it at your target .pdb")

    if not problems and (design_path.startswith("/content/drive/")
                         or starting_pdb.startswith("/content/drive/")):
        if not os.path.isdir("/content/drive/MyDrive"):
            problems.append("Google Drive is not mounted - run cell 2 first")

    if not problems and not os.path.isfile(starting_pdb):
        problems.append("starting_pdb does not exist: " + starting_pdb)

    if problems:
        for p in problems:
            print("PROBLEM: " + p)
        raise ValueError("Settings are not usable yet - see the " +
                         str(len(problems)) + " problem(s) printed above.")

    with open(starting_pdb) as f:
        pdb_lines = f.readlines()

    atoms = [l for l in pdb_lines if l.startswith("ATOM")]
    if not atoms:
        raise ValueError(starting_pdb + " has no ATOM records - is it really a PDB file?")

    # Read the structure the way BindCraft will see it: one entry per residue,
    # in file order, so the sequence and the numbering can be shown back to you.
    AA3TO1 = {
        "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C",
        "GLN": "Q", "GLU": "E", "GLY": "G", "HIS": "H", "ILE": "I",
        "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P",
        "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V",
        "MSE": "M", "SEC": "U", "PYL": "O",
    }

    chain_res = {}           # chain -> list of (residue id as written, 3-letter name)
    seen = set()
    for l in atoms:
        ch, rid, rname = l[21], l[22:27].strip(), l[17:20].strip()
        if (ch, rid) not in seen:
            seen.add((ch, rid))
            chain_res.setdefault(ch, []).append((rid, rname))

    # The whole of each selected chain is backpropagated through the network,
    # so asking for a chain that is not there fails late and expensively.
    wanted = [c.strip() for c in chains.split(",") if c.strip()]
    missing = [c for c in wanted if c not in chain_res]
    if missing:
        raise ValueError(
            "chain(s) " + ",".join(missing) + " are not in " + os.path.basename(starting_pdb)
            + ". It contains: " + ",".join(sorted(chain_res)))

    def resnum(rid):
        """Residue number without any insertion code."""
        digits = "".join(c for c in rid if c.isdigit() or c == "-")
        return int(digits) if digits else None

    # --- hotspots ---------------------------------------------------------
    # Accepts the formats upstream documents: A, A1-10, A31, 31, 2-10.
    # Checked against the structure now, because a hotspot that does not exist
    # is silently ignored during the run and you never find out.
    hotspots = []
    hotspot_errors = []
    for tok in [t.strip() for t in target_hotspot_residues.split(",") if t.strip()]:
        ch = tok[0] if tok[0].isalpha() else None
        body = tok[1:] if ch else tok
        targets = [ch] if ch else wanted
        if ch and ch not in chain_res:
            hotspot_errors.append(tok + ": chain " + ch + " is not in the structure")
            continue
        if ch and ch not in wanted:
            hotspot_errors.append(tok + ": chain " + ch + " is in the file but not in `chains`")
            continue
        if not body:                                   # a bare chain id: the whole chain
            hotspots += [(ch, rid, rn) for rid, rn in chain_res[ch]]
            continue
        if "-" in body:
            lo, hi = body.split("-", 1)
            lo, hi = int(lo), int(hi)
        else:
            lo = hi = int(body)
        found = [(c, rid, rn) for c in targets for rid, rn in chain_res[c]
                 if resnum(rid) is not None and lo <= resnum(rid) <= hi]
        if not found:
            hotspot_errors.append(tok + ": no such residue in chain(s) " + ",".join(targets))
        hotspots += found

    if hotspot_errors:
        for e in hotspot_errors:
            print("PROBLEM: " + e)
        raise ValueError("target_hotspot_residues does not match the structure - see above.")

    # --- the report -------------------------------------------------------
    bar = "=" * 68
    rep = [bar, "TARGET CHECK  -  " + os.path.basename(starting_pdb), bar]
    rep.append("chains in file:    " + ", ".join(sorted(chain_res)))
    rep.append("designing against: " + ", ".join(wanted))
    ignored = [c for c in sorted(chain_res) if c not in wanted]
    if ignored:
        rep.append("ignored:           " + ", ".join(ignored))
    rep.append("")

    for c in sorted(chain_res):
        res = chain_res[c]
        nums = [resnum(r) for r, _ in res if resnum(r) is not None]
        span = str(min(nums)) + "-" + str(max(nums)) if nums else "?"
        tag = "" if c in wanted else "   (not selected)"
        rep.append("chain " + c + ":  " + str(len(res)) + " residues, numbered " + span + tag)

        gaps = [(nums[i], nums[i + 1]) for i in range(len(nums) - 1)
                if nums[i + 1] - nums[i] > 1]
        if gaps:
            rep.append("   numbering gaps after: " +
                       ", ".join(str(a) + " (jumps to " + str(b) + ")" for a, b in gaps[:5]))
        unknown = sorted({rn for _, rn in res if rn not in AA3TO1})
        if unknown:
            rep.append("   non-standard residues shown as X: " + ", ".join(unknown[:8]))

        if c in wanted:
            seq = "".join(AA3TO1.get(rn, "X") for _, rn in res)
            for i in range(0, len(seq), 50):
                label = str(res[i][0]).rjust(5)
                line = " ".join(seq[j:j + 10] for j in range(i, min(i + 50, len(seq)), 10))
                rep.append("   " + label + "  " + line)
        rep.append("")

    if target_hotspot_residues.strip():
        rep.append("hotspots: " + target_hotspot_residues)
        shown = hotspots[:25]
        for c, rid, rn in shown:
            rep.append("   " + (c + rid).ljust(8) + rn + "  " + AA3TO1.get(rn, "X"))
        if len(hotspots) > len(shown):
            rep.append("   ... and " + str(len(hotspots) - len(shown)) + " more")
        aromatic = [h for h in hotspots if AA3TO1.get(h[2]) in "FYW"]
        greasy = [h for h in hotspots if AA3TO1.get(h[2]) in "ILMV"]
        rep.append("   " + str(len(hotspots)) + " residues, " + str(len(aromatic)) +
                   " aromatic (F/Y/W), " + str(len(greasy)) + " hydrophobic (I/L/M/V)")
        if not aromatic:
            rep.append("   note: no aromatic anchor - short peptides grip these patches poorly")
    else:
        rep.append("hotspots: none given - BindCraft will choose the site itself")
    rep.append("")

    rep.append("binder length:  " + lengths + " residues, sampled per trajectory")
    rep.append("designs wanted: " + str(number_of_final_designs))

    het = sorted({l[17:20].strip() for l in pdb_lines if l.startswith("HETATM")})
    if het:
        rep.append("")
        rep.append("WARNING: HETATM records present (" + ", ".join(het) + ").")
        rep.append("Waters, ligands and alternate conformations should be removed first.")
    rep.append(bar)
    target_report = rep

    lengths = [int(x.strip()) for x in lengths.split(',') if len(lengths.split(',')) == 2]

    if len(lengths) != 2:
        raise ValueError("Incorrect specification of binder lengths.")

    settings = {
        "design_path": design_path,
        "binder_name": binder_name,
        "starting_pdb": starting_pdb,
        "chains": chains,
        "target_hotspot_residues": target_hotspot_residues,
        "lengths": lengths,
        "number_of_final_designs": number_of_final_designs
    }

    target_settings_path = os.path.join(design_path, binder_name+".json")
    os.makedirs(design_path, exist_ok=True)

    with open(target_settings_path, 'w') as f:
        json.dump(settings, f, indent=4)

currenttime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Binder design settings updated at: {currenttime}")
print(f"New .json file with target settings has been generated in: {target_settings_path}")

# Shown last so it is what you are left looking at: check it before cell 10.
for line in globals().get("target_report", []):
    print(line)

# The run loop counts what is already in Accepted/ and stops once it has enough,
# which is what makes a repeat run look like it exited for no reason.
accepted_dir = os.path.join(design_path, "Accepted")
have = len([f for f in os.listdir(accepted_dir)
            if f.endswith(".pdb")]) if os.path.isdir(accepted_dir) else 0
if have:
    print("")
    print(f"{design_path} already holds {have} accepted design(s), and "
          f"number_of_final_designs is {number_of_final_designs}.")
    if have >= number_of_final_designs:
        print("  The next run will exit immediately. Raise the number, or point")
        print("  design_path at a new folder to start a fresh campaign.")
    else:
        print(f"  The next run will add up to {number_of_final_designs - have} more.")

In [ ]:
#@title 4. Advanced settings
# @markdown ---
# @markdown Which binder design protocol to run? Default is recommended. "Beta-sheet" promotes the design of more beta sheeted proteins, but requires more sampling. "Peptide" is optimised for helical peptide binders.
design_protocol = "Peptide" # @param ["Default","Beta-sheet","Peptide"]
# @markdown What prediction protocol to use?. "Default" performs single sequence prediction of the binder. "HardTarget" uses initial guess to improve complex prediction for difficult targets, but might introduce some bias.
prediction_protocol = "Default" # @param ["Default","HardTarget"]
# @markdown What interface design method to use?. "AlphaFold2" is the default, interface is generated by AlphaFold2. "MPNN" uses soluble MPNN to optimise the interface.
interface_protocol = "AlphaFold2" # @param ["AlphaFold2","MPNN"]
# @markdown What target template protocol to use? "Default" allows for limited amount flexibility. "Masked" allows for greater target flexibility on both sidechain and backbone level.
template_protocol = "Default" # @param ["Default","Masked"]
# @markdown ---

if design_protocol == "Default":
    design_protocol_tag = "default_4stage_multimer"
elif design_protocol == "Beta-sheet":
    design_protocol_tag = "betasheet_4stage_multimer"
elif design_protocol == "Peptide":
    design_protocol_tag = "peptide_3stage_multimer"
else:
    raise ValueError(f"Unsupported design protocol")

if interface_protocol == "AlphaFold2":
    interface_protocol_tag = ""
elif interface_protocol == "MPNN":
    interface_protocol_tag = "_mpnn"
else:
    raise ValueError(f"Unsupported interface protocol")

if template_protocol == "Default":
    template_protocol_tag = ""
elif template_protocol == "Masked":
    template_protocol_tag = "_flexible"
else:
    raise ValueError(f"Unsupported template protocol")

if design_protocol in ["Peptide"]:
    prediction_protocol_tag = ""
else:
    if prediction_protocol == "Default":
        prediction_protocol_tag = ""
    elif prediction_protocol == "HardTarget":
        prediction_protocol_tag = "_hardtarget"
    else:
        raise ValueError(f"Unsupported prediction protocol")

advanced_settings_path = "/content/bindcraft/settings_advanced/" + design_protocol_tag + interface_protocol_tag + template_protocol_tag + prediction_protocol_tag + ".json"

currenttime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Advanced design settings updated at: {currenttime}")

In [ ]:
#@title 5. Filters
# @markdown ---
# @markdown Which filters for designs to use? "Default" are recommended, "Peptide" are for the design of peptide binders, "Relaxed" are more permissive but may result in fewer experimental successes, "Peptide_Relaxed" are more permissive filters for non-helical peptides, "None" is for benchmarking.
filter_option = "Peptide" # @param ["Default", "Peptide", "Relaxed", "Peptide_Relaxed", "None"]
# @markdown ---

if filter_option == "Default":
    filter_settings_path = "/content/bindcraft/settings_filters/default_filters.json"
elif filter_option == "Peptide":
    filter_settings_path = "/content/bindcraft/settings_filters/peptide_filters.json"
elif filter_option == "Relaxed":
    filter_settings_path = "/content/bindcraft/settings_filters/relaxed_filters.json"
elif filter_option == "Peptide_Relaxed":
    filter_settings_path = "/content/bindcraft/settings_filters/peptide_relaxed_filters.json"
elif filter_option == "None":
    filter_settings_path = "/content/bindcraft/settings_filters/no_filters.json"
else:
    raise ValueError(f"Unsupported filter type")

currenttime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Filter settings updated at: {currenttime}")

# Everything is set, BindCraft is ready to run!


In [ ]:
# @title 6. Import functions and settings
from bindcraft.functions import *

args = {"settings":target_settings_path,
        "filters":filter_settings_path,
        "advanced":advanced_settings_path}

# Check if JAX-capable GPU is available, otherwise exit
check_jax_gpu()

# perform checks of input setting files
settings_path, filters_path, advanced_path = (args["settings"], args["filters"], args["advanced"])

### load settings from JSON
target_settings, advanced_settings, filters = load_json_settings(settings_path, filters_path, advanced_path)

settings_file = os.path.basename(settings_path).split('.')[0]
filters_file = os.path.basename(filters_path).split('.')[0]
advanced_file = os.path.basename(advanced_path).split('.')[0]

### load AF2 model settings
design_models, prediction_models, multimer_validation = load_af2_models(advanced_settings["use_multimer_design"])

### perform checks on advanced_settings
bindcraft_folder = "colab"
advanced_settings = perform_advanced_settings_check(advanced_settings, bindcraft_folder)

### generate directories, design path names can be found within the function
design_paths = generate_directories(target_settings["design_path"])

### generate dataframes
trajectory_labels, design_labels, final_labels = generate_dataframe_labels()

trajectory_csv = os.path.join(target_settings["design_path"], 'trajectory_stats.csv')
mpnn_csv = os.path.join(target_settings["design_path"], 'mpnn_design_stats.csv')
final_csv = os.path.join(target_settings["design_path"], 'final_design_stats.csv')
failure_csv = os.path.join(target_settings["design_path"], 'failure_csv.csv')

create_dataframe(trajectory_csv, trajectory_labels)
create_dataframe(mpnn_csv, design_labels)
create_dataframe(final_csv, final_labels)
generate_filter_pass_csv(failure_csv, args["filters"])

currenttime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Loaded design functions and settings at: {currenttime}")

In [ ]:
#@title 7. Initialise PyRosetta

####################################
####################################
####################################
### initialise PyRosetta
pr.init(f'-ignore_unrecognized_res -ignore_zero_occupancy -mute all -holes:dalphaball {advanced_settings["dalphaball_path"]} -corrections::beta_nov16 true -relax:default_repeats 1')

In [ ]:
#@title 8. Compatibility patches (required) { display-mode: "form" }
# @markdown Shims for API changes in current JAX / Python 3.13. **Required** -
# @markdown without these the pipeline cannot run. Safe to re-run any number of
# @markdown times. Must run AFTER cell 6, and again if you re-run cell 6.
import gc
import importlib
import sys

import jax
import jax.numpy as jnp
import numpy as _np


def _from_notebook(f):
    """True if f was defined in a notebook cell rather than in a package."""
    try:
        fn = f.__code__.co_filename
    except AttributeError:
        return False
    return ("ipykernel" in fn) or ("<ipython" in fn)


# ---- 1. jnp.clip ----------------------------------------------------------
# Modern JAX renamed a_min/a_max to min/max and made arr positional-only, but
# ColabDesign still calls jnp.clip(x, a_min=..., a_max=...).
# Capturing jnp.clip blindly would self-recurse if this cell ran twice, so we
# only ever wrap a function that came from the jax package.
_cur = jnp.clip
_real_clip = None
if getattr(_cur, "_bc_patched", False):
    _real_clip = getattr(_cur, "_bc_real", None)
elif not _from_notebook(_cur):
    _real_clip = _cur
if _real_clip is None or _from_notebook(_real_clip):
    for _modname in ("jax._src.numpy.lax_numpy", "jax._src.numpy.ufuncs"):
        try:
            _m = importlib.import_module(_modname)
        except Exception:
            continue
        _cand = getattr(_m, "clip", None)
        if _cand is not None and not _from_notebook(_cand):
            _real_clip = _cand
            break
if _real_clip is None:
    raise RuntimeError(
        "Could not find a clean jnp.clip. Runtime > Restart session, then run "
        "the cells in order."
    )


def _compat_clip(a=None, a_min=None, a_max=None, **kw):
    if "min" in kw:
        a_min = kw.pop("min")
    if "max" in kw:
        a_max = kw.pop("max")
    return _real_clip(a, a_min, a_max, **kw)


_compat_clip._bc_patched = True
_compat_clip._bc_real = _real_clip
jnp.clip = _compat_clip

# ---- 2. clear_mem ---------------------------------------------------------
# ColabDesign's clear_mem calls jax.lib.xla_bridge, which no longer exists.
# Defining it at notebook scope also replaces the binding that cell 6's
# `from bindcraft.functions import *` put in globals(), which is the one the
# run loop actually calls.
def clear_mem():
    jax.clear_caches()
    gc.collect()


clear_mem._bc_patched = True

_patched = 0
for _name, _m in list(sys.modules.items()):
    if _m is None:
        continue
    try:
        _f = getattr(_m, "clear_mem", None)
    except Exception:
        continue
    if callable(_f) and not getattr(_f, "_bc_patched", False):
        try:
            setattr(_m, "clear_mem", clear_mem)
            _patched += 1
        except Exception:
            pass

# ---- self-tests -----------------------------------------------------------
assert list(jnp.clip(_np.array([-5, 0, 5]), a_min=0, a_max=2)) == [0, 0, 2]
assert list(jnp.clip(_np.array([-5, 0, 5]), min=0, max=2)) == [0, 0, 2]
clear_mem()
print("jnp.clip shim OK (both a_min/a_max and min/max call styles)")
print("clear_mem replaced in " + str(_patched) + " module(s) + this notebook's namespace")


In [ ]:
#@title 9. Tuning controls { display-mode: "form" }
# @markdown ### Preset
# @markdown Pick a preset to **ignore the fields below** and use known-good values.
# @markdown Choose `Custom` to use whatever the fields are set to.
preset = "Production defaults"  # @param ["Production defaults", "Fast debug", "Custom - use fields below"]
# @markdown <small>`Production defaults` is the reset: whatever this protocol's own
# @markdown settings_advanced/*.json says, read back from the copy taken before any
# @markdown change, gates untouched. `Fast debug` cuts iterations ~10x and drops the pLDDT gate to
# @markdown 0.1 so every trajectory reaches the downstream code - fast, but the
# @markdown designs are structurally meaningless.</small>
# @markdown ---
# @markdown ### Custom values
# @markdown Only used when the preset above is `Custom`. The numbers shown are the
# @markdown Peptide protocol's values; Default and Beta-sheet use num_seqs 20, so
# @markdown prefer `Production defaults` over retyping them by hand.
# @markdown #### Trajectory optimisation &mdash; the biggest lever on runtime
soft_iterations = 75  # @param {type:"integer"}
temporary_iterations = 45  # @param {type:"integer"}
hard_iterations = 5  # @param {type:"integer"}
greedy_iterations = 15  # @param {type:"integer"}

# @markdown #### MPNN sequence design
num_seqs = 10  # @param {type:"integer"}
max_mpnn_sequences = 2  # @param {type:"integer"}

# @markdown #### AlphaFold2
num_recycles_design = 1  # @param {type:"integer"}
num_recycles_validation = 3  # @param {type:"integer"}
sample_models = True  # @param {type:"boolean"}

# @markdown #### Limits and output
max_trajectories = 0  # @param {type:"integer"}
# @markdown <small>0 means unlimited. It counts rows in trajectory_stats.csv, and
# @markdown rejected trajectories never write one, so it is no safeguard against a
# @markdown run that rejects everything.</small>
save_design_animations = True  # @param {type:"boolean"}
save_design_trajectory_plots = True  # @param {type:"boolean"}

# @markdown #### Acceptance gates (hardcoded in colabdesign_utils.py upstream)
trajectory_plddt_threshold = 0.7  # @param {type:"number"}
max_ca_clashes = 0  # @param {type:"integer"}
# @markdown <small>A trajectory is rejected below the pLDDT threshold, or above the
# @markdown CA-clash count. Lowering the gate lets unfolded junk reach MPNN - use it
# @markdown only to exercise the downstream pipeline, never for real designs.</small>

# ---------------------------------------------------------------------------
import importlib
import inspect
import os
import re
import shutil

if "advanced_settings" not in globals():
    raise RuntimeError("Run cell 6 (Import functions and settings) before this cell.")
if "clear_mem" not in globals() or not getattr(clear_mem, "_bc_patched", False):
    raise RuntimeError("Run cell 8 (Compatibility patches) before this cell.")

PRODUCTION = {
    "soft_iterations": 75,
    "temporary_iterations": 45,
    "hard_iterations": 5,
    "greedy_iterations": 15,
    "num_seqs": 10,
    "max_mpnn_sequences": 2,
    "num_recycles_design": 1,
    "num_recycles_validation": 3,
    "sample_models": True,
    "max_trajectories": 0,
    "save_design_animations": True,
    "save_design_trajectory_plots": True,
    "trajectory_plddt_threshold": 0.7,
    "max_ca_clashes": 0,
}

FAST_DEBUG = {
    "soft_iterations": 10,
    "temporary_iterations": 5,
    "hard_iterations": 2,
    "greedy_iterations": 0,
    "num_seqs": 2,
    "max_mpnn_sequences": 1,
    "num_recycles_design": 0,
    "num_recycles_validation": 1,
    "sample_models": False,
    "max_trajectories": 2,
    "save_design_animations": False,
    "save_design_trajectory_plots": False,
    "trajectory_plddt_threshold": 0.1,
    "max_ca_clashes": 0,
}

# The stock values for THIS protocol, captured before anything is changed.
# settings_advanced/*.json differs per protocol - num_seqs is 20 for Default and
# Beta-sheet but 10 for Peptide - so a hardcoded table would quietly run a
# non-stock value whenever the protocol changed. Re-snapshotted if cell 4 is
# pointed at a different JSON.
_key = globals().get("advanced_settings_path", "?")
if globals().get("_bc_stock_key") != _key:
    _bc_stock_settings = dict(advanced_settings)
    _bc_stock_key = _key

# Keys absent from the JSON (the two gates) fall back to the upstream source values
STOCK = {k: _bc_stock_settings.get(k, PRODUCTION[k]) for k in PRODUCTION}
if STOCK["max_trajectories"] in (False, None):
    STOCK["max_trajectories"] = 0

if preset.startswith("Production"):
    cfg = dict(STOCK)
elif preset.startswith("Fast"):
    cfg = dict(FAST_DEBUG)
else:
    cfg = {k: globals()[k] for k in PRODUCTION}

# keep the plain variables in step with cfg, so later cells (e.g. 14's check on
# save_design_animations) see what was actually applied rather than the field
for _k, _v in cfg.items():
    globals()[_k] = _v

print("Preset: " + preset)
print("Stock values for this protocol: " + os.path.basename(_key))
_drift = [k for k in STOCK if cfg[k] != STOCK[k]]
if _drift:
    print("Differs from this protocol's stock settings in "
          + str(len(_drift)) + " value(s):")
    for _k in _drift:
        print("   " + _k + ": " + str(STOCK[_k]) + " -> " + str(cfg[_k]))
else:
    print("All values are at this protocol's stock settings.")
print()

# --- the JSON-backed settings ---------------------------------------------
_json_keys = [k for k in PRODUCTION
              if k not in ("trajectory_plddt_threshold", "max_ca_clashes")]
_wanted = {k: cfg[k] for k in _json_keys}
_wanted["max_trajectories"] = (
    False if cfg["max_trajectories"] in (0, None) else cfg["max_trajectories"])

_changed, _same, _absent = [], 0, []
for _k, _v in _wanted.items():
    if _k not in advanced_settings:
        _absent.append(_k)
    elif advanced_settings[_k] != _v:
        _changed.append((_k, advanced_settings[_k], _v))
        advanced_settings[_k] = _v
    else:
        _same += 1

print("advanced_settings: " + str(len(_changed)) + " changed, "
      + str(_same) + " already as requested")
for _k, _old, _newv in _changed:
    print("   " + _k + ": " + str(_old) + " -> " + str(_newv))
if _absent:
    print("   not in this protocol's JSON, ignored: " + ", ".join(_absent))
print()

# --- the two gates that upstream hardcodes in the source file -------------
_CDU = "/content/bindcraft/functions/colabdesign_utils.py"
_BAK = _CDU + ".orig"
if not os.path.isfile(_CDU):
    print("   SKIPPED gates - " + _CDU + " not found; has cell 1 run?")
else:
    if not os.path.isfile(_BAK):
        shutil.copy2(_CDU, _BAK)
    # always rewrite from the pristine backup so repeated runs never compound
    with open(_BAK) as _fh:
        _orig = _fh.read()
    _new, _n1 = re.subn(r"(final_plddt\s*<\s*)0\.7\b",
                        r"\g<1>" + str(cfg["trajectory_plddt_threshold"]), _orig)
    _new, _n2 = re.subn(r"(ca_clashes\s*>\s*)0\b",
                        r"\g<1>" + str(cfg["max_ca_clashes"]), _new)
    if _n1 == 0:
        print("   WARNING - 'final_plddt < 0.7' not found; upstream may have changed")
    if _n2 == 0:
        print("   WARNING - 'ca_clashes > 0' not found; upstream may have changed")
    with open(_CDU, "w") as _fh:
        _fh.write(_new)

    _cdu = importlib.reload(
        importlib.import_module("bindcraft.functions.colabdesign_utils"))
    binder_hallucination = _cdu.binder_hallucination  # rebind for the run loop
    if not getattr(getattr(_cdu, "clear_mem", None), "_bc_patched", False):
        _cdu.clear_mem = clear_mem  # the reload re-imported the broken one

    _live = inspect.getsource(_cdu.binder_hallucination)
    _m1 = re.search(r"final_plddt\s*<\s*([0-9.]+)", _live)
    _m2 = re.search(r"ca_clashes\s*>\s*(\d+)", _live)
    print("   live pLDDT gate: " + (_m1.group(0) if _m1 else "NOT FOUND"))
    print("   live clash gate: " + (_m2.group(0) if _m2 else "NOT FOUND"))

print()
if cfg["trajectory_plddt_threshold"] < 0.7 or cfg["max_ca_clashes"] > 0:
    print("*** GATES RELAXED - designs from this run are NOT meaningful. ***")
elif _drift:
    print("Mode: MODIFIED (gates intact, so designs remain valid)")
else:
    print("Mode: STOCK - identical to upstream BindCraft for this protocol")


In [ ]:
#@title 10. Run BindCraft!
####################################
###################### BindCraft Run
####################################
# Colab-specific: live displays
num_sampled_trajectories = len(pd.read_csv(trajectory_csv))
num_accepted_designs = len(pd.read_csv(final_csv))
sampled_trajectories_label = HTML(value=f"<h3 style='color: #1f77b4;'>Sampled Trajectories: <span style='color: #1f77b4;'>{num_sampled_trajectories}</span></h3>")
accepted_designs_label = HTML(value=f"<h3 style='color: #2ca02c;'>Accepted Designs: <span style='color: #2ca02c;'>{num_accepted_designs}</span></h3>")
display(VBox([sampled_trajectories_label, accepted_designs_label]))

# initialise counters
script_start_time = time.time()
trajectory_n = 1
accepted_designs = 0

### start design loop
while True:
    ### check if we have the target number of binders
    final_designs_reached = check_accepted_designs(design_paths, mpnn_csv, final_labels, final_csv, advanced_settings, target_settings, design_labels)

    if final_designs_reached:
        # stop design loop execution
        break

    ### check if we reached maximum allowed trajectories
    max_trajectories_reached = check_n_trajectories(design_paths, advanced_settings)

    if max_trajectories_reached:
        break

    ### Initialise design
    # measure time to generate design
    trajectory_start_time = time.time()

    # generate random seed to vary designs
    seed = int(np.random.randint(0, high=999999, size=1, dtype=int)[0])

    # sample binder design length randomly from defined distribution
    samples = np.arange(min(target_settings["lengths"]), max(target_settings["lengths"]) + 1)
    length = np.random.choice(samples)

    # load desired helicity value to sample different secondary structure contents
    helicity_value = load_helicity(advanced_settings)

    # generate design name and check if same trajectory was already run
    design_name = target_settings["binder_name"] + "_l" + str(length) + "_s"+ str(seed)
    trajectory_dirs = ["Trajectory", "Trajectory/Relaxed", "Trajectory/LowConfidence", "Trajectory/Clashing"]
    trajectory_exists = any(os.path.exists(os.path.join(design_paths[trajectory_dir], design_name + ".pdb")) for trajectory_dir in trajectory_dirs)

    if not trajectory_exists:
        print("Starting trajectory: "+design_name)

        ### Begin binder hallucination
        trajectory = binder_hallucination(design_name, target_settings["starting_pdb"], target_settings["chains"],
                                            target_settings["target_hotspot_residues"], length, seed, helicity_value,
                                            design_models, advanced_settings, design_paths, failure_csv)
        trajectory_metrics = copy_dict(trajectory._tmp["best"]["aux"]["log"]) # contains plddt, ptm, i_ptm, pae, i_pae
        trajectory_pdb = os.path.join(design_paths["Trajectory"], design_name + ".pdb")

        # round the metrics to two decimal places
        trajectory_metrics = {k: round(v, 2) if isinstance(v, float) else v for k, v in trajectory_metrics.items()}

        # time trajectory
        trajectory_time = time.time() - trajectory_start_time
        trajectory_time_text = f"{'%d hours, %d minutes, %d seconds' % (int(trajectory_time // 3600), int((trajectory_time % 3600) // 60), int(trajectory_time % 60))}"
        print("Starting trajectory took: "+trajectory_time_text)
        print("")

        # Proceed if there is no trajectory termination signal
        if trajectory.aux["log"]["terminate"] == "":
            # Relax binder to calculate statistics
            trajectory_relaxed = os.path.join(design_paths["Trajectory/Relaxed"], design_name + ".pdb")
            pr_relax(trajectory_pdb, trajectory_relaxed)

            # define binder chain, placeholder in case multi-chain parsing in ColabDesign gets changed
            binder_chain = "B"

            # Calculate clashes before and after relaxation
            num_clashes_trajectory = calculate_clash_score(trajectory_pdb)
            num_clashes_relaxed = calculate_clash_score(trajectory_relaxed)

            # secondary structure content of starting trajectory binder and interface
            trajectory_alpha, trajectory_beta, trajectory_loops, trajectory_alpha_interface, trajectory_beta_interface, trajectory_loops_interface, trajectory_i_plddt, trajectory_ss_plddt = calc_ss_percentage(trajectory_pdb, advanced_settings, binder_chain)

            # analyze interface scores for relaxed af2 trajectory
            trajectory_interface_scores, trajectory_interface_AA, trajectory_interface_residues = score_interface(trajectory_relaxed, binder_chain)

            # starting binder sequence
            trajectory_sequence = trajectory.get_seq(get_best=True)[0]

            # analyze sequence
            traj_seq_notes = validate_design_sequence(trajectory_sequence, num_clashes_relaxed, advanced_settings)

            # target structure RMSD compared to input PDB
            trajectory_target_rmsd = unaligned_rmsd(target_settings["starting_pdb"], trajectory_pdb, target_settings["chains"], 'A')

            # save trajectory statistics into CSV
            trajectory_data = [design_name, advanced_settings["design_algorithm"], length, seed, helicity_value, target_settings["target_hotspot_residues"], trajectory_sequence, trajectory_interface_residues,
                                trajectory_metrics['plddt'], trajectory_metrics['ptm'], trajectory_metrics['i_ptm'], trajectory_metrics['pae'], trajectory_metrics['i_pae'],
                                trajectory_i_plddt, trajectory_ss_plddt, num_clashes_trajectory, num_clashes_relaxed, trajectory_interface_scores['binder_score'],
                                trajectory_interface_scores['surface_hydrophobicity'], trajectory_interface_scores['interface_sc'], trajectory_interface_scores['interface_packstat'],
                                trajectory_interface_scores['interface_dG'], trajectory_interface_scores['interface_dSASA'], trajectory_interface_scores['interface_dG_SASA_ratio'],
                                trajectory_interface_scores['interface_fraction'], trajectory_interface_scores['interface_hydrophobicity'], trajectory_interface_scores['interface_nres'], trajectory_interface_scores['interface_interface_hbonds'],
                                trajectory_interface_scores['interface_hbond_percentage'], trajectory_interface_scores['interface_delta_unsat_hbonds'], trajectory_interface_scores['interface_delta_unsat_hbonds_percentage'],
                                trajectory_alpha_interface, trajectory_beta_interface, trajectory_loops_interface, trajectory_alpha, trajectory_beta, trajectory_loops, trajectory_interface_AA, trajectory_target_rmsd,
                                trajectory_time_text, traj_seq_notes, settings_file, filters_file, advanced_file]
            insert_data(trajectory_csv, trajectory_data)

            if advanced_settings["enable_mpnn"]:
                # initialise MPNN counters
                mpnn_n = 1
                accepted_mpnn = 0
                mpnn_dict = {}
                design_start_time = time.time()

                ### MPNN redesign of starting binder
                mpnn_trajectories = mpnn_gen_sequence(trajectory_pdb, binder_chain, trajectory_interface_residues, advanced_settings)
                existing_mpnn_sequences = set(pd.read_csv(mpnn_csv, usecols=['Sequence'])['Sequence'].values)

                # create set of MPNN sequences with allowed amino acid composition
                restricted_AAs = set(aa.strip().upper() for aa in advanced_settings["omit_AAs"].split(',')) if advanced_settings["force_reject_AA"] else set()

                mpnn_sequences = sorted({
                    mpnn_trajectories['seq'][n][-length:]: {
                        'seq': mpnn_trajectories['seq'][n][-length:],
                        'score': mpnn_trajectories['score'][n],
                        'seqid': mpnn_trajectories['seqid'][n]
                    } for n in range(advanced_settings["num_seqs"])
                    if (not restricted_AAs or not any(aa in mpnn_trajectories['seq'][n][-length:].upper() for aa in restricted_AAs))
                    and mpnn_trajectories['seq'][n][-length:] not in existing_mpnn_sequences
                }.values(), key=lambda x: x['score'])

                del existing_mpnn_sequences

                # check whether any sequences are left after amino acid rejection and duplication check, and if yes proceed with prediction
                if mpnn_sequences:
                    # add optimisation for increasing recycles if trajectory is beta sheeted
                    if advanced_settings["optimise_beta"] and float(trajectory_beta) > 15:
                        advanced_settings["num_recycles_validation"] = advanced_settings["optimise_beta_recycles_valid"]

                    ### Compile prediction models once for faster prediction of MPNN sequences
                    clear_mem()
                    # compile complex prediction model
                    complex_prediction_model = mk_afdesign_model(protocol="binder", num_recycles=advanced_settings["num_recycles_validation"], data_dir=advanced_settings["af_params_dir"],
                                                                use_multimer=multimer_validation)
                    complex_prediction_model.prep_inputs(pdb_filename=target_settings["starting_pdb"], chain=target_settings["chains"], binder_len=length, rm_target_seq=advanced_settings["rm_template_seq_predict"],
                                                        rm_target_sc=advanced_settings["rm_template_sc_predict"])

                    # compile binder monomer prediction model
                    binder_prediction_model = mk_afdesign_model(protocol="hallucination", use_templates=False, initial_guess=False,
                                                                use_initial_atom_pos=False, num_recycles=advanced_settings["num_recycles_validation"],
                                                                data_dir=advanced_settings["af_params_dir"], use_multimer=multimer_validation)
                    binder_prediction_model.prep_inputs(length=length)

                    # iterate over designed sequences
                    for mpnn_sequence in mpnn_sequences:
                        mpnn_time = time.time()

                        # generate mpnn design name numbering
                        mpnn_design_name = design_name + "_mpnn" + str(mpnn_n)
                        mpnn_score = round(mpnn_sequence['score'],2)
                        mpnn_seqid = round(mpnn_sequence['seqid'],2)

                        # add design to dictionary
                        mpnn_dict[mpnn_design_name] = {'seq': mpnn_sequence['seq'], 'score': mpnn_score, 'seqid': mpnn_seqid}

                        # save fasta sequence
                        if advanced_settings["save_mpnn_fasta"] is True:
                            save_fasta(mpnn_design_name, mpnn_sequence['seq'], design_paths)

                        ### Predict mpnn redesigned binder complex using masked templates
                        mpnn_complex_statistics, pass_af2_filters = predict_binder_complex(complex_prediction_model,
                                                                                        mpnn_sequence['seq'], mpnn_design_name,
                                                                                        target_settings["starting_pdb"], target_settings["chains"],
                                                                                        length, trajectory_pdb, prediction_models, advanced_settings,
                                                                                        filters, design_paths, failure_csv)

                        # if AF2 filters are not passed then skip the scoring
                        if not pass_af2_filters:
                            print(f"Base AF2 filters not passed for {mpnn_design_name}, skipping interface scoring")
                            mpnn_n += 1
                            continue

                        # calculate statistics for each model individually
                        for model_num in prediction_models:
                            mpnn_design_pdb = os.path.join(design_paths["MPNN"], f"{mpnn_design_name}_model{model_num+1}.pdb")
                            mpnn_design_relaxed = os.path.join(design_paths["MPNN/Relaxed"], f"{mpnn_design_name}_model{model_num+1}.pdb")

                            if os.path.exists(mpnn_design_pdb):
                                # Calculate clashes before and after relaxation
                                num_clashes_mpnn = calculate_clash_score(mpnn_design_pdb)
                                num_clashes_mpnn_relaxed = calculate_clash_score(mpnn_design_relaxed)

                                # analyze interface scores for relaxed af2 trajectory
                                mpnn_interface_scores, mpnn_interface_AA, mpnn_interface_residues = score_interface(mpnn_design_relaxed, binder_chain)

                                # secondary structure content of starting trajectory binder
                                mpnn_alpha, mpnn_beta, mpnn_loops, mpnn_alpha_interface, mpnn_beta_interface, mpnn_loops_interface, mpnn_i_plddt, mpnn_ss_plddt = calc_ss_percentage(mpnn_design_pdb, advanced_settings, binder_chain)

                                # unaligned RMSD calculate to determine if binder is in the designed binding site
                                rmsd_site = unaligned_rmsd(trajectory_pdb, mpnn_design_pdb, binder_chain, binder_chain)

                                # calculate RMSD of target compared to input PDB
                                target_rmsd = target_pdb_rmsd(mpnn_design_pdb, target_settings["starting_pdb"], target_settings["chains"])

                                # add the additional statistics to the mpnn_complex_statistics dictionary
                                mpnn_complex_statistics[model_num+1].update({
                                    'i_pLDDT': mpnn_i_plddt,
                                    'ss_pLDDT': mpnn_ss_plddt,
                                    'Unrelaxed_Clashes': num_clashes_mpnn,
                                    'Relaxed_Clashes': num_clashes_mpnn_relaxed,
                                    'Binder_Energy_Score': mpnn_interface_scores['binder_score'],
                                    'Surface_Hydrophobicity': mpnn_interface_scores['surface_hydrophobicity'],
                                    'ShapeComplementarity': mpnn_interface_scores['interface_sc'],
                                    'PackStat': mpnn_interface_scores['interface_packstat'],
                                    'dG': mpnn_interface_scores['interface_dG'],
                                    'dSASA': mpnn_interface_scores['interface_dSASA'],
                                    'dG/dSASA': mpnn_interface_scores['interface_dG_SASA_ratio'],
                                    'Interface_SASA_%': mpnn_interface_scores['interface_fraction'],
                                    'Interface_Hydrophobicity': mpnn_interface_scores['interface_hydrophobicity'],
                                    'n_InterfaceResidues': mpnn_interface_scores['interface_nres'],
                                    'n_InterfaceHbonds': mpnn_interface_scores['interface_interface_hbonds'],
                                    'InterfaceHbondsPercentage': mpnn_interface_scores['interface_hbond_percentage'],
                                    'n_InterfaceUnsatHbonds': mpnn_interface_scores['interface_delta_unsat_hbonds'],
                                    'InterfaceUnsatHbondsPercentage': mpnn_interface_scores['interface_delta_unsat_hbonds_percentage'],
                                    'InterfaceAAs': mpnn_interface_AA,
                                    'Interface_Helix%': mpnn_alpha_interface,
                                    'Interface_BetaSheet%': mpnn_beta_interface,
                                    'Interface_Loop%': mpnn_loops_interface,
                                    'Binder_Helix%': mpnn_alpha,
                                    'Binder_BetaSheet%': mpnn_beta,
                                    'Binder_Loop%': mpnn_loops,
                                    'Hotspot_RMSD': rmsd_site,
                                    'Target_RMSD': target_rmsd
                                })

                                # save space by removing unrelaxed predicted mpnn complex pdb?
                                if advanced_settings["remove_unrelaxed_complex"]:
                                    os.remove(mpnn_design_pdb)

                        # calculate complex averages
                        mpnn_complex_averages = calculate_averages(mpnn_complex_statistics, handle_aa=True)

                        ### Predict binder alone in single sequence mode
                        binder_statistics = predict_binder_alone(binder_prediction_model, mpnn_sequence['seq'], mpnn_design_name, length,
                                                                trajectory_pdb, binder_chain, prediction_models, advanced_settings, design_paths)

                        # extract RMSDs of binder to the original trajectory
                        for model_num in prediction_models:
                            mpnn_binder_pdb = os.path.join(design_paths["MPNN/Binder"], f"{mpnn_design_name}_model{model_num+1}.pdb")

                            if os.path.exists(mpnn_binder_pdb):
                                rmsd_binder = unaligned_rmsd(trajectory_pdb, mpnn_binder_pdb, binder_chain, "A")

                            # append to statistics
                            binder_statistics[model_num+1].update({
                                    'Binder_RMSD': rmsd_binder
                                })

                            # save space by removing binder monomer models?
                            if advanced_settings["remove_binder_monomer"]:
                                os.remove(mpnn_binder_pdb)

                        # calculate binder averages
                        binder_averages = calculate_averages(binder_statistics)

                        # analyze sequence to make sure there are no cysteins and it contains residues that absorb UV for detection
                        seq_notes = validate_design_sequence(mpnn_sequence['seq'], mpnn_complex_averages.get('Relaxed_Clashes', None), advanced_settings)

                        # measure time to generate design
                        mpnn_end_time = time.time() - mpnn_time
                        elapsed_mpnn_text = f"{'%d hours, %d minutes, %d seconds' % (int(mpnn_end_time // 3600), int((mpnn_end_time % 3600) // 60), int(mpnn_end_time % 60))}"


                        # Insert statistics about MPNN design into CSV, will return None if corresponding model does note exist
                        model_numbers = range(1, 6)
                        statistics_labels = ['pLDDT', 'pTM', 'i_pTM', 'pAE', 'i_pAE', 'i_pLDDT', 'ss_pLDDT', 'Unrelaxed_Clashes', 'Relaxed_Clashes', 'Binder_Energy_Score', 'Surface_Hydrophobicity',
                                            'ShapeComplementarity', 'PackStat', 'dG', 'dSASA', 'dG/dSASA', 'Interface_SASA_%', 'Interface_Hydrophobicity', 'n_InterfaceResidues', 'n_InterfaceHbonds', 'InterfaceHbondsPercentage',
                                            'n_InterfaceUnsatHbonds', 'InterfaceUnsatHbondsPercentage', 'Interface_Helix%', 'Interface_BetaSheet%', 'Interface_Loop%', 'Binder_Helix%',
                                            'Binder_BetaSheet%', 'Binder_Loop%', 'InterfaceAAs', 'Hotspot_RMSD', 'Target_RMSD']

                        # Initialize mpnn_data with the non-statistical data
                        mpnn_data = [mpnn_design_name, advanced_settings["design_algorithm"], length, seed, helicity_value, target_settings["target_hotspot_residues"], mpnn_sequence['seq'], mpnn_interface_residues, mpnn_score, mpnn_seqid]

                        # Add the statistical data for mpnn_complex
                        for label in statistics_labels:
                            mpnn_data.append(mpnn_complex_averages.get(label, None))
                            for model in model_numbers:
                                mpnn_data.append(mpnn_complex_statistics.get(model, {}).get(label, None))

                        # Add the statistical data for binder
                        for label in ['pLDDT', 'pTM', 'pAE', 'Binder_RMSD']:  # These are the labels for binder alone
                            mpnn_data.append(binder_averages.get(label, None))
                            for model in model_numbers:
                                mpnn_data.append(binder_statistics.get(model, {}).get(label, None))

                        # Add the remaining non-statistical data
                        mpnn_data.extend([elapsed_mpnn_text, seq_notes, settings_file, filters_file, advanced_file])

                        # insert data into csv
                        insert_data(mpnn_csv, mpnn_data)

                        # find best model number by pLDDT
                        plddt_values = {i: mpnn_data[i] for i in range(11, 15) if mpnn_data[i] is not None}

                        # Find the key with the highest value
                        highest_plddt_key = int(max(plddt_values, key=plddt_values.get))

                        # Output the number part of the key
                        best_model_number = highest_plddt_key - 10
                        best_model_pdb = os.path.join(design_paths["MPNN/Relaxed"], f"{mpnn_design_name}_model{best_model_number}.pdb")

                        # run design data against filter thresholds
                        filter_conditions = check_filters(mpnn_data, design_labels, filters)
                        if filter_conditions == True:
                            print(mpnn_design_name+" passed all filters")
                            accepted_mpnn += 1
                            accepted_designs += 1

                            # copy designs to accepted folder
                            shutil.copy(best_model_pdb, design_paths["Accepted"])

                            # insert data into final csv
                            final_data = [''] + mpnn_data
                            insert_data(final_csv, final_data)

                            # copy animation from accepted trajectory
                            if advanced_settings["save_design_animations"]:
                                accepted_animation = os.path.join(design_paths["Accepted/Animation"], f"{design_name}.html")
                                if not os.path.exists(accepted_animation):
                                    shutil.copy(os.path.join(design_paths["Trajectory/Animation"], f"{design_name}.html"), accepted_animation)

                            # copy plots of accepted trajectory
                            plot_files = os.listdir(design_paths["Trajectory/Plots"])
                            plots_to_copy = [f for f in plot_files if f.startswith(design_name) and f.endswith('.png')]
                            for accepted_plot in plots_to_copy:
                                source_plot = os.path.join(design_paths["Trajectory/Plots"], accepted_plot)
                                target_plot = os.path.join(design_paths["Accepted/Plots"], accepted_plot)
                                if not os.path.exists(target_plot):
                                    shutil.copy(source_plot, target_plot)

                        else:
                            print(f"Unmet filter conditions for {mpnn_design_name}")
                            failure_df = pd.read_csv(failure_csv)
                            special_prefixes = ('Average_', '1_', '2_', '3_', '4_', '5_')
                            incremented_columns = set()

                            for column in filter_conditions:
                                base_column = column
                                for prefix in special_prefixes:
                                    if column.startswith(prefix):
                                        base_column = column.split('_', 1)[1]

                                if base_column not in incremented_columns:
                                    failure_df[base_column] = failure_df[base_column] + 1
                                    incremented_columns.add(base_column)

                            failure_df.to_csv(failure_csv, index=False)
                            shutil.copy(best_model_pdb, design_paths["Rejected"])

                        # increase MPNN design number
                        mpnn_n += 1

                        # if enough mpnn sequences of the same trajectory pass filters then stop
                        if accepted_mpnn >= advanced_settings["max_mpnn_sequences"]:
                            break

                    if accepted_mpnn >= 1:
                        print("Found "+str(accepted_mpnn)+" MPNN designs passing filters")
                    else:
                        print("No accepted MPNN designs found for this trajectory.")

                else:
                    print('Duplicate MPNN designs sampled with different trajectory, skipping current trajectory optimisation')

                # save space by removing unrelaxed design trajectory PDB
                if advanced_settings["remove_unrelaxed_trajectory"]:
                    os.remove(trajectory_pdb)

                # measure time it took to generate designs for one trajectory
                design_time = time.time() - design_start_time
                design_time_text = f"{'%d hours, %d minutes, %d seconds' % (int(design_time // 3600), int((design_time % 3600) // 60), int(design_time % 60))}"
                print("Design and validation of trajectory "+design_name+" took: "+design_time_text)

            # analyse the rejection rate of trajectories to see if we need to readjust the design weights
            if trajectory_n >= advanced_settings["start_monitoring"] and advanced_settings["enable_rejection_check"]:
                acceptance = accepted_designs / trajectory_n
                if not acceptance >= advanced_settings["acceptance_rate"]:
                    print("The ratio of successful designs is lower than defined acceptance rate! Consider changing your design settings!")
                    print("Script execution stopping...")
                    break

        # increase trajectory number
        trajectory_n += 1

        # Colab-specific: update counters
        num_sampled_trajectories = len(pd.read_csv(trajectory_csv))
        num_accepted_designs = len(pd.read_csv(final_csv))
        sampled_trajectories_label.value = f"Sampled trajectories: {num_sampled_trajectories}"
        accepted_designs_label.value = f"Accepted designs: {num_accepted_designs}"

### Script finished
elapsed_time = time.time() - script_start_time
elapsed_text = f"{'%d hours, %d minutes, %d seconds' % (int(elapsed_time // 3600), int((elapsed_time % 3600) // 60), int(elapsed_time % 60))}"
print("Finished all designs. Script execution for "+str(trajectory_n)+" trajectories took: "+elapsed_text)

In [ ]:
#@title 11. Consolidate & Rank Designs
# @markdown Fixed: the upstream version assigned the filename stem to
# @markdown target_settings["binder_name"] - the very key it then compares
# @markdown against - which silently breaks every rank past the first and raises
# @markdown ValueError on any filename without "_model".
import os
import shutil

import pandas as pd

accepted_binders = [f for f in os.listdir(design_paths["Accepted"]) if f.endswith(".pdb")]

for _f in os.listdir(design_paths["Accepted/Ranked"]):
    os.remove(os.path.join(design_paths["Accepted/Ranked"], _f))

design_df = pd.read_csv(mpnn_csv).sort_values("Average_i_pTM", ascending=False)
final_df = pd.DataFrame(columns=final_labels)

_skipped = []
rank = 1
for _, row in design_df.iterrows():
    for binder in accepted_binders:
        if "_model" not in binder:
            if binder not in _skipped:
                _skipped.append(binder)
            continue
        design_stem, model = binder.rsplit("_model", 1)
        if design_stem != row["Design"]:
            continue

        row_data = {"Rank": rank, **{label: row[label] for label in design_labels}}
        final_df = pd.concat([final_df, pd.DataFrame([row_data])], ignore_index=True)

        shutil.copyfile(
            os.path.join(design_paths["Accepted"], binder),
            os.path.join(
                design_paths["Accepted/Ranked"],
                str(rank) + "_" + design_stem + "_model"
                + model.rsplit(".", 1)[0] + ".pdb",
            ),
        )
        rank += 1
        break

final_df.to_csv(final_csv, index=False)

print("Ranked " + str(len(final_df)) + " design(s) -> " + str(final_csv))
if _skipped:
    print("Skipped " + str(len(_skipped)) + " file(s) with no '_model' in the name: "
          + str(_skipped))
assert "_model" not in str(target_settings["binder_name"]), (
    "binder_name got clobbered: " + str(target_settings["binder_name"]))
print("binder_name intact: " + str(target_settings["binder_name"]))


In [ ]:
#@title 12. Top 20 Designs
df = pd.read_csv(os.path.join(design_path, 'final_design_stats.csv'))
df.head(20)

In [ ]:
#@title 13. Top Design Display
# @markdown Fixed: guards against an empty Ranked/ folder instead of raising
# @markdown IndexError on glob(...)[0].
import glob
import os

import py3Dmol

_dir = os.path.join(design_path, "Accepted", "Ranked")
_hits = sorted(glob.glob(os.path.join(_dir, "1_*.pdb")))

if not _hits:
    print("No ranked design in " + _dir)
    print("Run cell 11 first. If it reported 0 designs, nothing has passed the "
          "filters yet.")
else:
    print("Showing: " + os.path.basename(_hits[0]))
    view = py3Dmol.view()
    with open(_hits[0]) as _fh:
        view.addModel(_fh.read(), "pdb")
    view.setBackgroundColor("white")
    view.setStyle({"chain": "A"}, {"cartoon": {"color": "#3c5b6f"}})
    view.setStyle({"chain": "B"}, {"cartoon": {"color": "#B76E79"}})
    view.zoomTo()
    view.show()


In [ ]:
#@title 14. Display animation
# @markdown Fixed: the upstream version passed a file *path* to HTML() instead of
# @markdown the file's contents, and crashed when nothing had been ranked.
import glob
import os

from IPython.display import HTML, display

_dir = os.path.join(design_path, "Accepted", "Ranked")
_hits = sorted(glob.glob(os.path.join(_dir, "1_*.pdb")))

if not _hits:
    print("No ranked design found - run cell 11 first.")
elif not globals().get("save_design_animations", True):
    print("Animations were turned off in cell 9, so none were written.")
else:
    _name = os.path.basename(_hits[0]).split("1_", 1)[1].split("_mpnn")[0]
    _path = os.path.join(design_path, "Accepted", "Animation", _name + ".html")
    if os.path.isfile(_path):
        with open(_path) as _fh:
            display(HTML(_fh.read()))
    else:
        print("Animation not found: " + _path)
        _zips = glob.glob(os.path.join(design_path, "Accepted", "Animation*.zip"))
        _zips += glob.glob(os.path.join(design_path, "Trajectory", "Animation*.zip"))
        if _zips:
            print("BindCraft's zip_animations setting archived them:")
            for _z in _zips:
                print("   " + _z)
            print("Unzip one of those to view it.")


In [ ]:
#@title 15. Diagnose rejected trajectories { display-mode: "form" }
# @markdown Why trajectories are being rejected, and *where* the clashes are.
# @markdown
# @markdown The gate that prints "Severe clashes detected" counts any two CA atoms
# @markdown closer than 2.5 A anywhere in the complex (same residue and directly
# @markdown sequential pairs excluded). Because it looks at CA only, it does not
# @markdown skip same-chain pairs, so a collapsed binder backbone trips it just as
# @markdown readily as a binder driven into the target. This cell says which.
max_pdbs = 6  # @param {type:"integer"}

import glob
import math
import os
from itertools import combinations

import pandas as pd

# ---- 0. will the next run actually do anything? ---------------------------
# The loop calls check_accepted_designs() before sampling: it counts .pdb files
# in Accepted/ and, if that count has reached number_of_final_designs, it reranks
# and breaks. Nothing is sampled, and the message it prints names the count it
# found rather than your target, which reads as if the target were that number.
_acc_dir = os.path.join(design_path, "Accepted")
_have = len([f for f in os.listdir(_acc_dir)
             if f.endswith(".pdb") and not f.startswith(".")])         if os.path.isdir(_acc_dir) else 0
_want = int(target_settings["number_of_final_designs"])         if "target_settings" in globals() else None

print("Accepted designs on disk: " + str(_have)
      + ("   target: " + str(_want) if _want is not None else ""))
if _want is not None and _have >= _want:
    print("   -> cell 10 will rerank and exit in seconds without designing")
    print("      anything. To make new ones, raise number_of_final_designs in")
    print("      cell 3 to more than " + str(_have) + ", then re-run, in order:")
    print("      cell 3 (writes the settings JSON), cell 6 (reloads it),")
    print("      cells 8 and 9 (cell 6 re-imports the broken clear_mem), cell 10.")
    print("      Re-running cell 10 alone keeps the old target and exits again.")
elif _want is not None:
    print("   -> cell 10 has " + str(_want - _have) + " more to find, it will run")
print()

# ---- 1. the tally BindCraft already keeps ---------------------------------
if "failure_csv" in globals() and os.path.isfile(failure_csv):
    _f = pd.read_csv(failure_csv)
    _tally = {c: int(_f[c].sum()) for c in _f.columns if _f[c].dtype != object}
    _tally = {k: v for k, v in sorted(_tally.items(), key=lambda kv: -kv[1]) if v}
    print("Rejection tally so far")
    for _k, _v in _tally.items():
        print("   " + _k + ": " + str(_v))
    if not _tally:
        print("   nothing recorded yet")
else:
    print("No failure CSV found - has cell 10 run?")
print()

# ---- 2. where the clashes actually are ------------------------------------
_target_chains = {c.strip() for c in str(globals().get("chains", "A")).split(",")
                  if c.strip()}

def _read_ca(pdb):
    out = []
    with open(pdb) as fh:
        for l in fh:
            if l.startswith("ATOM") and l[12:16].strip() == "CA":
                try:
                    b = float(l[60:66])
                except ValueError:
                    b = 0.0
                out.append((l[21], int(l[22:26]), float(l[30:38]),
                            float(l[38:46]), float(l[46:54]), b))
    return out

def _find_clashes(atoms, cutoff=2.5):
    hits = []
    for a, b in combinations(atoms, 2):
        if a[0] == b[0] and abs(a[1] - b[1]) <= 1:
            continue
        d = math.dist(a[2:5], b[2:5])
        if d < cutoff:
            hits.append((a, b, d))
    return hits

_pdbs, _seen = [], set()
for _d in ("Trajectory/Clashing", "Trajectory/LowConfidence", "Trajectory"):
    for _p in sorted(glob.glob(os.path.join(design_path, _d, "*.pdb"))):
        if os.path.basename(_p) not in _seen:
            _seen.add(os.path.basename(_p))
            _pdbs.append(_p)

_totals = {"target-target": 0, "binder-binder": 0, "binder-target": 0}
_plddts = []

if not _pdbs:
    print("No trajectory PDBs under " + os.path.join(design_path, "Trajectory"))
else:
    for _p in _pdbs[:max_pdbs]:
        _at = _read_ca(_p)
        _chains = sorted({a[0] for a in _at})
        _binder = [c for c in _chains if c not in _target_chains]
        _hits = _find_clashes(_at)
        _kinds = {"target-target": 0, "binder-binder": 0, "binder-target": 0}
        for _a, _b, _d in _hits:
            _ta, _tb = _a[0] in _target_chains, _b[0] in _target_chains
            _k = ("target-target" if _ta and _tb
                  else "binder-binder" if not _ta and not _tb else "binder-target")
            _kinds[_k] += 1
            _totals[_k] += 1

        _pl = {}
        for _c in _chains:
            _v = [a[5] for a in _at if a[0] == _c]
            _pl[_c] = (sum(_v) / len(_v)) / (100.0 if max(_v) > 1.5 else 1.0)
        for _c in _binder:
            _plddts.append(_pl[_c])

        print(os.path.relpath(_p, design_path))
        print("   chains " + str(_chains) + "  binder=" + str(_binder or "?")
              + "   " + ", ".join("pLDDT " + c + "=" + format(_pl[c], ".2f")
                                  for c in _chains))
        print("   CA pairs under 2.5 A: " + str(len(_hits)) + "   " + str(_kinds))
        for _a, _b, _d in _hits[:4]:
            print("      " + _a[0] + str(_a[1]) + " - " + _b[0] + str(_b[1])
                  + "   " + format(_d, ".2f") + " A")
        print()

    # ---- 3. verdict --------------------------------------------------------
    print("-" * 68)
    _worst = max(_totals, key=_totals.get) if any(_totals.values()) else None
    if _worst == "target-target":
        print("Clashes are inside the TARGET itself, so no binder can ever pass.")
        print("Clean the target PDB: one model only, no altlocs, no duplicated or")
        print("overlapping residue numbering, then re-run from cell 3.")
    elif _worst == "binder-binder":
        print("Clashes are inside the BINDER - its own backbone is folding through")
        print("itself. Short binders do this: the optimiser has too few residues to")
        print("satisfy the contact loss without collapsing. Try a longer range in")
        print("cell 3 (for example 40,60) before touching anything else.")
    elif _worst == "binder-target":
        print("The BINDER is being driven INTO the target. Usually the hotspots are")
        print("pointing it at a buried or crowded site - check they are surface")
        print("residues on the chain you are targeting.")
    else:
        print("No CA clashes in the files inspected.")

    if _plddts:
        _m = sum(_plddts) / len(_plddts)
        print()
        print("Mean binder pLDDT across these trajectories: " + format(_m, ".2f"))
        if _m < 0.7:
            print("Below the hardcoded 0.7 gate, so even with the clashes fixed")
            print("these trajectories would be rejected as low confidence. That gate")
            print("runs before your Filters setting, which is why Filters=None does")
            print("not let them through.")
